# Convert Model

In [2]:
import tensorflow as tf
from tensorflow import keras

# load model
model = keras.models.load_model("export/original_model.keras")
model.summary()

2024-11-11 21:08:19.875720: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731355699.894086    9432 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731355699.899457    9432 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-11 21:08:19.917485: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-11 21:08:21.882565: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 185,120 (723.13 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 123,414 (482.09 KB)

In [3]:
model.export("export/tmp")

INFO:tensorflow:Assets written to: export/tmp/assets


INFO:tensorflow:Assets written to: export/tmp/assets


Saved artifact at 'export/tmp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  128263561924432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561924048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561925392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561926544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561927888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561928464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561927120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561929616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561928656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561930000: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [4]:
!python -m tf2onnx.convert --keras export/original_model.keras --output export/original_model.onnx

2024-11-11 21:08:23.242957: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731355703.261466   14087 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731355703.267034   14087 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-11 21:08:23.285350: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to ex

In [31]:
import numpy as np

def convert_data(x_data: np.ndarray, y_data: np.ndarray):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    return (x_data, y_data)


# for now we are just interested in the training data and not the test data
(X_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)

images = tf.cast(X_train, tf.float32) / 255.0
mnist_ds = tf.data.Dataset.from_tensor_slices((images)).batch(1)
def representative_data_gen():
    for input_value in mnist_ds.take(100):
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.experimental_new_converter = True

converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

open("export/quantized_model.tflite", "wb").write(tflite_quant_model)


INFO:tensorflow:Assets written to: /tmp/tmpmvyi3mvp/assets


INFO:tensorflow:Assets written to: /tmp/tmpmvyi3mvp/assets


Saved artifact at '/tmp/tmpmvyi3mvp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  128263561924432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561924048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561925392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561926544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561927888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561928464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561927120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561929616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561928656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128263561930000: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/markus/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1731356979.883404    9432 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1731356979.883421    9432 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2024-11-11 21:29:39.883618: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpmvyi3mvp
2024-11-11 21:29:39.884254: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2024-11-11 21:29:39.884264: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpmvyi3mvp
2024-11-11 21:29:39.889414: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2024-11-11 21:29:39.919479: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpmvyi3

70896

In [32]:
!python -m tf2onnx.convert --tflite export/quantized_model.tflite --output export/quantized_model.onnx

2024-11-11 21:29:42.790245: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731356982.810385   16688 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731356982.816482   16688 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-11 21:29:42.835873: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to ex